In [ ]:
from sklearn.cluster import AgglomerativeClustering

def drop_correlated_features( X , threshold = 0.9 ):
    """
    Args:
        - X (pd.DataFrame) : n,p feature matrix
        - threshold (float) : absolute correlation threshold group variables
    
    Returns:
        - pd.DataFrame : X with only the selected variables
        - dict : keys are the selected features , values are the list of features in the corresponding feature cluster 
    """
    
    corr_threshold = 0.9

    metric = 1 - X.corr().abs()

    HC = AgglomerativeClustering( n_clusters=None , metric='precomputed', linkage = 'single' , distance_threshold = (1-corr_threshold) )
    HC.fit(metric)

    variable_clusters = pd.Series( HC.labels_  , index = X.columns)

    cluster_to_features = variable_clusters.index.groupby(variable_clusters)

    ## keys are the selected feature in the cluster, values are the list of features in the cluster
    selected_features_to_features = { v[0]:list(v) for v in cluster_to_features.values() }

    return X.loc[:,selected_features_to_features.keys()] ,  selected_features_to_features 

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectPercentile
import numpy as np

## loading data
df_xpr = pd.read_csv("../data/TGCA_BRCA_expression_matrix.TPM.csv.gz" , index_col = 0)

df_clinical = pd.read_csv("../data/TGCA_BRCA_clinical_filtered.small.csv",index_col=0)
df_clinical = pd.get_dummies( df_clinical , drop_first=True)

## y is the poor_diagnosis
y = df_clinical.poor_prognosis

## ensuring the expression data is properly ordered
X_xpr = df_xpr.loc[ :, df_clinical.index].transpose() 

## selecting top 1% most variable genes
VT = SelectPercentile( score_func = lambda x,_ : np.var(x , axis = 0) ,
                       percentile = 1
                     )

X = pd.DataFrame( VT.fit_transform(X_xpr), columns=VT.get_feature_names_out() , index = X_xpr.index )
X , features_to_features_cluster = drop_correlated_features( X , threshold = 0.9 )


# adding age and sex to the set of features
X = pd.concat( [ df_clinical[['demographic.days_to_birth','demographic.sex_at_birth_male']] , X ] , axis=1 )

X.shape

# L1 regression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score




In [ ]:
%%time
feature_names = X.columns

logCs = []

coef_dict = {'name' : [],
             'val' : [],
             'log_C' : []}
accuracies = []

for C in np.logspace(-3,0,100):

    ppl = Pipeline([('scale' , StandardScaler()),
                ('model' , LogisticRegression(l1_ratio=1.0,
                                              C=C,
                                              solver="liblinear"))
               ])
    
    logCs.append(np.log10(C))
    accuracies.append( cross_val_score( ppl , X , y , scoring = 'accuracy').mean() )

    ppl.fit(X , y)
    
    coef_dict['name'] += list( feature_names )
    coef_dict['val'] += list( ppl['model'].coef_[0] )
    coef_dict['log_C'] += [np.log10(C)]* len(feature_names )

coef_df = pd.DataFrame(coef_dict)

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns

bestC = logCs[ np.argmax( accuracies ) ]

fig,ax = plt.subplots(1,2,figsize = (15,7))

ax[0].plot(logCs , accuracies)
ax[0].set_xlabel("log10( C )")
ax[0].set_ylabel("cross-validated accuracy")
ax[0].axvline( bestC, color='r', ls = '--' )

sns.lineplot( x = 'log_C' , y='val' , hue = 'name' , data= coef_df , ax = ax[1] , legend=None)
ax[1].axvline( bestC , color='r', ls = '--' )
ax[1].set_ylabel( 'coefficient' )


fig.suptitle("logistic regression with an L1 regularization.")


If we use a more classical grid-search approach:

In [ ]:
%%time
from sklearn.model_selection import GridSearchCV

pipeline_lr=Pipeline([('scalar',StandardScaler()), 
                      ('model',LogisticRegression(l1_ratio=1.0,
                                                  solver="liblinear"))])

grid_values = {'model__C': np.logspace(-2,0,50) }
# define the hyperparameters you want to test
# with the range over which you want it to be tested.

# Feed it to the GridSearchCV 
grid_lr = GridSearchCV(pipeline_lr, 
                       param_grid = grid_values, 
                       scoring='accuracy',
                       cv=5, 
                       n_jobs=-1)


## this cell throws a lot of warning, I remove them with the lines under
grid_lr.fit(X, y)

print(f"best cross-validated {grid_lr.scoring} : {grid_lr.best_score_:.2f}")
for k,v in grid_lr.best_params_.items():
    print(f'\t{k} : {v}')

In [ ]:
from operator import itemgetter

featureW = pd.DataFrame( {
                          'feature':X.columns,
                          'weight':grid_lr.best_estimator_['model'].coef_[0]
                         } )

featureWsorted = featureW.sort_values(by=['weight'] , 
                                      ascending=False , 
                                      key=lambda col : col.abs())

# get the non-null ones
print('Features sorted per importance:')
print( featureWsorted.loc[ featureWsorted["weight"] !=0 ] )

## exercise :

1. build a reduced data set from the set of feature selected with L1
2. Use cross-validation to evaluate the accuracy of a logistic regression pipeline **without penalisation** on the reduced dataset
3. Do you get a better or a worse cross-validated accuracy than with the penalized model on the full data?

In [ ]:
selected_features = featureWsorted.loc[ featureWsorted["weight"] !=0 , 'feature']

# ... your code here ...

--- **correction** ---

In [ ]:
# %load solution_L1.py

## other models

L1 has more or less similar hyper-parameters in some models.

For example, in decision tree models and random forest you can use `max_depth` and [cost-complexity pruning](https://scikit-learn.org/stable/auto_examples/tree/plot_cost_complexity_pruning.html#sphx-glr-auto-examples-tree-plot-cost-complexity-pruning-py) (which removes the parts of trees which do not add much information)

In [ ]:
from sklearn.ensemble import RandomForestClassifier


rf = RandomForestClassifier(n_jobs=-1, max_depth = 2 , n_estimators= 20, random_state=1443)

## difficult to decide on a value of ccp_alpha (which is linked to pruning)
## so we use CV to find a good value
rf_grid = GridSearchCV(rf,
                       {"ccp_alpha" : np.logspace(-6,-3,20)}, # post
                       scoring='accuracy'
                      )
%time rf_grid.fit(X,y)

In [ ]:
feature_importances = pd.Series(rf_grid.best_estimator_.feature_importances_ , 
                                index = X.columns)

print(f'features with non-zero importance: {(feature_importances!=0).sum()}')

feature_importances[feature_importances!=0]

## exercise:

How many features are selected both by the L1-penalized logistic regression and the random forest?

--- **correction** ---

In [ ]:
# %load solution_l1_RF_intersection.py